# Cold Start Chat
## Chat system for addressing the cold start problem head on.
This will be using NLP to take in a user's input and parse any relevant information to then add to the users profile, e.g. they like horror films, they don't like Brad Pitt, etc.
This is to give them an immediate set of recommendations based on their profiles.

Another objective in these questions is to not ask redundant ones, e.g. if they've already said they don't like non-English films, we shouldn't ask them if they like French films.


### Libraries
- Recommendation library: LensKit
- NLP library: spaCy

In [2]:
print("hello world")

hello world


In [2]:
# from cgitb import small

import numpy as np
import lenskit
import pandas as pd
import spacy

import os # accessing directory structure

import ast

# from mpl_toolkits.mplot3d import Axes3D
# from sklearn.preprocessing import StandardScaler
# import matplotlib.pyplot as plt # plotting



In [4]:
# Define new storage path
new_actors_path = os.path.expanduser("~/datasets/actors/")

In [5]:
# I only want the primary name from the actors dataset - column primaryName
actors = pd.read_csv(f"{new_actors_path}1/combined.csv")
actors.head(2)

"""
Fred Astaire,1899,1987.0,"actor,miscellaneous,producer",The Towering Inferno
Lauren Bacall,1924,2014.0,"actress,soundtrack,archive_footage",To Have and Have Not
"""

actorsPrimaryName = actors['primaryName']
actorsPrimaryName.head(2)


0     Fred Astaire
1    Lauren Bacall
Name: primaryName, dtype: object

In [6]:


print("NumPy version:", np.__version__)
print("LensKit version:", lenskit.__version__)
print("Pandas version:", pd.__version__)
print("spaCy version:", spacy.__version__)

NumPy version: 1.26.4
LensKit version: 0.14.4
Pandas version: 2.2.3
spaCy version: 3.8.3


In [5]:
# print the head of the dataset
movies_path = os.path.expanduser("~/datasets/movies/7")
ratings = pd.read_csv(f"{movies_path}/ratings.csv")
movies = pd.read_csv(f"{movies_path}/movies_metadata.csv")






/var/folders/lr/44zcpqfx2196g_qy5t3kc7m40000gp/T/ipykernel_22805/4083713369.py:4: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies = pd.read_csv(f"{movies_path}/movies_metadata.csv")


In [3]:
movies_path = os.path.expanduser("~/datasets/movies/7")



movies = pd.read_csv(f"{movies_path}/movies_metadata.csv", usecols=['id', 'original_title', 'release_date'])
movies.head(47000)

,id,original_title,release_date
0,862,Toy Story,1995-10-30
1,8844,Jumanji,1995-12-15
2,15602,Grumpier Old Men,1995-12-22
3,31357,Waiting to Exhale,1995-12-22
4,11862,Father of the Bride Part II,1995-02-10
...,...,...,...
45461,439050,رگ خواب,NaN
45462,111109,Siglo ng Pagluluwal,2011-11-17
45463,67758,Betrayal,2003-08-01
45464,227506,Satana likuyushchiy,1917-10-21


In [11]:
movies.shape

(45466, 3)

In [8]:
print("Ratings dataset:")
ratings.head(2)

Ratings dataset:


,userId,movieId,rating,timestamp
0,1,110,1.0,1425941529
1,1,147,4.5,1425942435


In [9]:
print("Movies dataset:")
movies.head(2)

Movies dataset:


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0


In [8]:
print("Movies metadata:")
movies_metadata = pd.read_csv(f"{movies_path}/movies_metadata.csv")
movies_metadata.head(10)

Movies metadata:


/var/folders/lr/44zcpqfx2196g_qy5t3kc7m40000gp/T/ipykernel_22805/2455604186.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_metadata = pd.read_csv(f"{movies_path}/movies_metadata.csv")


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0
5,False,NaN,60000000,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...",NaN,949,tt0113277,en,Heat,"Obsessive master thief, Neil McCauley leads a ...",...,1995-12-15,187436818.0,170.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,A Los Angeles Crime Saga,Heat,False,7.7,1886.0
6,False,NaN,58000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 10749, '...",NaN,11860,tt0114319,en,Sabrina,An ugly duckling having undergone a remarkable...,...,1995-12-15,0.0,127.0,"[{'iso_639_1': 'fr', 'name': 'Français'}, {'is...",Released,You are cordially invited to the most surprisi...,Sabrina,False,6.2,141.0
7,False,NaN,0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",NaN,45325,tt0112302,en,Tom and Huck,"A mischievous young boy, Tom Sawyer, witnesses...",...,1995-12-22,0.0,97.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,The Original Bad Boys.,Tom and Huck,False,5.4,45.0
8,False,NaN,35000000,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",NaN,9091,tt0114576,en,Sudden Death,International action superstar Jean Claude Van...,...,1995-12-22,64350171.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Terror goes into overtime.,Sudden Death,False,5.5,174.0
9,False,"{'id': 645, 'name': 'James Bond Collection', '...",58000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",http://www.mgm.com/view/movie/757/Goldeneye/,710,tt0113189,en,GoldenEye,James Bond must unmask the mysterious head of ...,...,1995-11-16,352194034.0,130.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,No limits. No fears. No substitutes.,GoldenEye,False,6.6,1194.0


In [9]:
# print a list of movies_metadata columns
movies_metadata.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='object')

In [4]:
import pandas as pd
import os
print("Credits dataset:")
movies_path = os.path.expanduser("~/datasets/movies/7")
film_credits = pd.read_csv(f"{movies_path}/credits.csv")
film_credits.head(20)

# """
# [
# {'credit_id': '52fe43c59251416c7501d6f3', 'department': 'Directing', 'gender': 2, 'id': 1152, 'job': 'Director', 'name': 'Oliver Stone', 'profile_path': '/uHdNGBkrI74eYfUP2Uie7nuo0Nn.jpg'}, 
# {'credit_id': '52fe43c59251416c7501d6f9', 'department': 'Writing', 'gender': 2, 'id': 1152, 'job': 'Screenplay', 'name': 'Oliver Stone', 'profile_path': '/uHdNGBkrI74eYfUP2Uie7nuo0Nn.jpg'}, 
# {'credit_id': '52fe43c59251416c7501d6ff', 'department': 'Production', 'gender': 2, 'id': 5379, 'job': 'Producer', 'name': 'Dan Halsted', 'profile_path': None}, 
# {'credit_id': '52fe43c59251416c7501d705', 'department': 'Sound', 'gender': 2, 'id': 491, 'job': 'Original Music Composer', 'name': 'John Williams', 'profile_path': '/2Ats98PB1SH2yfEPikiLdhRuXZm.jpg'}, 
# {'credit_id': '52fe43c59251416c7501d70b', 'department': 'Writing', 'gender': 2, 'id': 17786, 'job': 'Screenplay', 'name': 'Stephen J. Rivele', 'profile_path': None}, 
# {'credit_id': '52fe43c59251416c7501d711', 'department': 'Camera', 'gender': 2, 'id': 149, 'job': 'Director of Photography', 'name': 'Robert Richardson', 'profile_path': '/9tLrDHL9qL9yYIhcJVaINwnzmWN.jpg'}, 
# {'credit_id': '52fe43c59251416c7501d717', 'department': 'Editing', 'gender': 2, 'id': 3189, 'job': 'Editor', 'name': 'Brian Berdan', 'profile_path': None}, 
# {'credit_id': '52fe43c59251416c7501d71d', 'department': 'Production', 'gender': 2, 'id': 1152, 'job': 'Producer', 'name': 'Oliver Stone', 'profile_path': '/uHdNGBkrI74eYfUP2Uie7nuo0Nn.jpg'}]
# """

Credits dataset:


,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862
5,"[{'cast_id': 25, 'character': 'Lt. Vincent Han...","[{'credit_id': '52fe4292c3a36847f802916d', 'de...",949
6,"[{'cast_id': 1, 'character': 'Linus Larrabee',...","[{'credit_id': '52fe44959251416c75039da9', 'de...",11860
7,"[{'cast_id': 2, 'character': 'Tom Sawyer', 'cr...","[{'credit_id': '52fe46bdc3a36847f810f797', 'de...",45325
8,"[{'cast_id': 1, 'character': 'Darren Francis T...","[{'credit_id': '52fe44dbc3a36847f80ae0f1', 'de...",9091
9,"[{'cast_id': 1, 'character': 'James Bond', 'cr...","[{'credit_id': '52fe426ec3a36847f801e14b', 'de...",710


In [14]:
film_credits['crew'].at[2]

"[{'credit_id': '52fe466a9251416c75077a89', 'department': 'Directing', 'gender': 2, 'id': 26502, 'job': 'Director', 'name': 'Howard Deutch', 'profile_path': '/68Vae1HkU1NxQZ6KEmuxIpno7c9.jpg'}, {'credit_id': '52fe466b9251416c75077aa3', 'department': 'Writing', 'gender': 2, 'id': 16837, 'job': 'Characters', 'name': 'Mark Steven Johnson', 'profile_path': '/6trChNn3o2bi4i2ipgMEAytwmZp.jpg'}, {'credit_id': '52fe466b9251416c75077aa9', 'department': 'Writing', 'gender': 2, 'id': 16837, 'job': 'Writer', 'name': 'Mark Steven Johnson', 'profile_path': '/6trChNn3o2bi4i2ipgMEAytwmZp.jpg'}, {'credit_id': '5675eb4b92514179dd003933', 'department': 'Crew', 'gender': 2, 'id': 1551320, 'job': 'Sound Recordist', 'name': 'Jack Keller', 'profile_path': None}]"

In [12]:
film_credits['crew'].head(5)

0    [{'credit_id': '52fe4284c3a36847f8024f49', 'de...
1    [{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...
2    [{'credit_id': '52fe466a9251416c75077a89', 'de...
3    [{'credit_id': '52fe44779251416c91011acb', 'de...
4    [{'credit_id': '52fe44959251416c75039ed7', 'de...
Name: crew, dtype: object

In [13]:
# grab all the people with a job of Director
# Credits format: columns: cast, crew, id  
# crew is a list of dictionaries - {'credit_id': '52fe43c59251416c7501d6f3', 'department': 'Directing', 'gender': 2, 'id': 1152, 'job': 'Director', 'name': 'Oliver Stone', 'profile_path': '/uHdNGBkrI74eYfUP2Uie7nuo0Nn.jpg'}, 
# each dictionary has keys: credit_id  
# department
import json
# [{'credit_id': '52fe4284c3a36847f8024f49', 'department': 'Directing', 'gender': 2, 'id': 7879, 'job': 'Director', 'name': 'John Lasseter', 'profile_path': '/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg'}, {'credit_id': '52fe4284c3a36847f8024f4f', 'department': 'Writing', 'gender': 2, 'id': 12891, 'job': 'Screenplay', 'name': 'Joss Whedon', 'profile_path': '/dTiVsuaTVTeGmvkhcyJvKp2A5kr.jpg'}, {'credit_id': '52fe4284c3a36847f8024f55', 'department': 'Writing', 'gender': 2, 'id': 7, 'job': 'Screenplay', 'name': 'Andrew Stanton', 'profile_path': '/pvQWsu0qc8JFQhMVJkTHuexUAa1.jpg'}, {'credit_id': '52fe4284c3a36847f8024f5b', 'department': 'Writing', 'gender': 2, 'id': 12892, 'job': 'Screenplay', 'name': 'Joel Cohen', 'profile_path': '/dAubAiZcvKFbboWlj7oXOkZnTSu.jpg'}, {'credit_id': '52fe4284c3a36847f8024f61', 'department': 'Writing', 'gender': 0, 'id':

# Sample data


# Function to filter directors
def filter_directors(crew_list):
    return [member for member in crew_list if member['job'] == 'Director']

# Apply the function to the 'crew' column
directors = film_credits['crew'].apply(lambda x: filter_directors(ast.literal_eval(x)))





# Function to extract directors
# def extract_directors(crew_list):
#     directors = []
#     try:
#         # directors = [member['name'] for member in crew_list if member['job'] == 'Director']
#         for member in crew_list:
#             print("member112")
#             print(member)
#             # if member['job'] == 'Director':
#             #     directors.append(member['name'])
#     except KeyError as e:
#         print(f"KeyError: {e} - Check if the key exists in the dictionary")
#     return directors



# type(credits)
# type(credits['crew'])


In [14]:
directors.head(5)

0    [{'credit_id': '52fe4284c3a36847f8024f49', 'de...
1    [{'credit_id': '52fe44bfc3a36847f80a7c7d', 'de...
2    [{'credit_id': '52fe466a9251416c75077a89', 'de...
3    [{'credit_id': '52fe44779251416c91011acb', 'de...
4    [{'credit_id': '52fe44959251416c75039eef', 'de...
Name: crew, dtype: object

In [15]:
# print the name of the director
# small_directors = directors.head(5)
# type(small_directors)

# take a list of director names from the pandas.core.series.Series of directors in directors
# Extract director names from the Series
director_names = directors.apply(lambda x: [d['name'] for d in x if 'name' in d])

# Flatten the list of lists into a single list
flat_director_names = [name for sublist in director_names for name in sublist]

print(flat_director_names)

['John Lasseter', 'Joe Johnston', 'Howard Deutch', 'Forest Whitaker', 'Charles Shyer', 'Michael Mann', 'Sydney Pollack', 'Peter Hewitt', 'Peter Hyams', 'Martin Campbell', 'Rob Reiner', 'Mel Brooks', 'Simon Wells', 'Oliver Stone', 'Renny Harlin', 'Martin Scorsese', 'Ang Lee', 'Allison Anders', 'Alexandre Rockwell', 'Robert Rodriguez', 'Quentin Tarantino', 'Steve Oedekerk', 'Joseph Ruben', 'Barry Sonnenfeld', 'Jon Amiel', 'Richard Donner', 'Victor Salva', 'Mike Figgis', 'Oliver Parker', 'Lesli Linka Glatter', 'Roger Michell', 'Jean-Pierre Jeunet', 'Marc Caro', 'Zhang Yimou', 'John N. Smith', 'Terry Gilliam', 'Jean-Jacques Annaud', 'Chris Noonan', 'Christopher Hampton', 'Tim Robbins', 'Stephen Low', 'Andy Tennant', 'Amy Heckerling', 'Darrell James Roodt', 'Richard Loncraine', 'Albert Hughes', 'Allen Hughes', 'Michael Hoffman', 'Paul W.S. Anderson', 'Gus Van Sant', 'Jocelyn Moorhouse', 'David Fincher', 'Mike Gabriel', 'Eric Goldberg', 'Patricia Rozema', 'Bryan Singer', 'Richard W. Munchkin

In [16]:

# Example data
data = [
    ("crew", "[{'credit_id': '52fe4284c3a36847f8024f49', 'department': 'Directing', 'gender': 2, 'id': 7879, 'job': 'Director', 'name': 'John Lasseter', 'profile_path': '/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg'}, {'credit_id': '52fe4284c3a36847f8024f4f', 'department': 'Writing', 'gender': 2, 'id': 12891, 'job': 'Screenplay', 'name': 'Joss Whedon', 'profile_path': '/dTiVsuaTVTeGmvkhcyJvKp2A5kr.jpg'}, {'credit_id': '589216f39251412dc2009cf3', 'department': 'Production', 'gender': 0, 'id': 84493, 'job': 'ADR Voice Casting', 'name': 'Mickie McGowan', 'profile_path': '/k7TjJBfINsg8vLQxJwos6XObAD6.jpg'}]")
]

# Create a DataFrame
df = pd.DataFrame(data, columns=['column_name', 'crew_data'])

# Parse the string representation of the list of dictionaries
df['crew_data'] = df['crew_data'].apply(lambda x: ast.literal_eval(x))

# Extract the director names
def extract_directors(crew_list):
    return [person['name'] for person in crew_list if person['job'] == 'Director']

df['directors'] = df['crew_data'].apply(extract_directors)

# Flatten the results
director_names = df['directors'].explode().dropna().tolist()

print(director_names)


['John Lasseter']


In [17]:
nlp = spacy.load('en_core_web_sm')



In [29]:

# def initialize_user_profile():
user_profile = {
    'films_liked': [],
    'films_disliked': [],
    'genres_positive': [],
    'genres_negative': [],
    'actors_positive': [],
    'actors_negative': [],
    'directors_positive': [],
    'directors_negative': [],
    'languages_positive': [],
    'languages_negative': []
}
    

In [28]:
# initialize_user_profile()

In [20]:
from transformers import pipeline, AutoTokenizer, TFAutoModelForSequenceClassification


In [22]:
# Load Twitter-roBERTa-base for Sentiment Analysis
tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment")
model = TFAutoModelForSequenceClassification.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment")
sentiment_analyzer = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)



All model checkpoint layers were used when initializing TFRobertaForSequenceClassification.

All the layers of TFRobertaForSequenceClassification were initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaForSequenceClassification for predictions without further training.
Device set to use 0


In [33]:

def parse_user_sentiment_input(user_input):
    """
    Parse user input for relevant information about likes and dislikes.
    """
    doc = nlp(user_input)
    entities = {'films': [], 'genres': [], 'actors': [], 'directors': [], 'languages': []}
    
    # Extract entities using spaCy's NER
    # for ent in doc.ents:
    #     if ent.label_ == 'PERSON':
    #         entities['actors'].append(ent.text)
    #     elif ent.label_ == 'WORK_OF_ART':
    #         entities['films'].append(ent.text)
    #     elif ent.label_ == 'LANGUAGE':
    #         entities['languages'].append(ent.text)
        
        # Add more entity types as needed
    
    # # Analyze sentiment for the entire sentence
    # sentences = [sent.text for sent in doc.sents]
    # for sentence in sentences:
    #     sentiment = sentiment_analyzer(sentence)[0]
    #     for key in entities.keys():
    #         for i, entity in enumerate(entities[key]):
    #             if entity in sentence:
    #                 if sentiment['label'] == 'NEGATIVE':
    #                     entities[key][i] = (entity, 'dislike')
    #                 elif sentiment['label'] == 'POSITIVE':
    #                     entities[key][i] = (entity, 'like')

    # Analyze sentiment all entities
    for key in entities.keys():
        for i, entity in enumerate(entities[key]):
            print("entity" + entity)
            sentiment = sentiment_analyzer(entity[0])[0]
            if sentiment['label'] == 'NEGATIVE':
                entities[key][i] = (False, entity)
            elif sentiment['label'] == 'POSITIVE':
                entities[key][i] = (True, 'like')

    return entities


    

In [ ]:
# Sentiment analysis for an entire sentence, topic agnostic
def sentiment_analyzer(sentence):
    """
    Analyze sentiment for a given sentence.
    """
    sentiment = sentiment_analyzer(sentence)[0]
    return sentiment


In [ ]:
def topic_parse_user_input(user_input, topic):
    """
    Parse user input for relevant information about likes and dislikes.
    """
    doc = nlp(user_input)
    entities = {'films': [], 'genres': [], 'actors': [], 'directors': [], 'languages': []}
    
    # Extract entities using spaCy's NER
    for ent in doc.ents:
        if ent.label_ == 'PERSON':
            entities[topic].append(ent.text)
        elif ent.label_ == 'WORK_OF_ART':
            entities[topic].append(ent.text)
        elif ent.label_ == 'LANGUAGE':
            entities[topic].append(ent.text)
        
        # Add more entity types as needed
    
    # Analyze sentiment for the entire sentence
    sentences = [sent.text for sent in doc.sents]
    for sentence in sentences:
        sentiment = sentiment_analyzer(sentence)[0]
        for key in entities.keys():
            for i, entity in enumerate(entities[key]):
                if entity in sentence:
                    if sentiment['label'] == 'NEGATIVE':
                        entities[key][i] = (entity, 'dislike')
                    elif sentiment['label'] == 'POSITIVE':
                        entities[key][i] = (entity, 'like')

    return entities

In [ ]:
# def update_user_profile(func_user_profile, func_parsed_data):
#     """
#     Update the user profile based on parsed data.
#     """
#     
#     for key in func_parsed_data.keys():
#         for entity in func_parsed_data[key]:
#             print(entity)
#             func_user_profile[key + '_' + entity + 'd'].append(entity[0][0])
#             # user_profile[key + '_' + entity + 'd'].append(entity[0][0])
#             
#     return func_user_profile

In [ ]:
def update_user_profile(question_topic):
    doc = nlp(user_input_string)
    sentences = [sent.text for sent in doc.sents]
    for sentence in sentences:
        sentiment = sentiment_analyzer(sentence)[0]
        user_profile[question_topic].append(sentiment['label'])
    

In [ ]:
user_input_string = "I like horror films. I like bullet train. I hate Brad Pitt. I hate French films. I like English films."


In [34]:


user_input_string = "I like horror films. I like bullet train. I hate Brad Pitt. I hate French films. I like English films."


# Parse user input
parsed_data = parse_user_sentiment_input(user_input_string)

# initialize_user_profile()
# Update the user profile
user_profile = update_user_profile(user_profile, parsed_data)

print("Updated User Profile:")
print(user_profile)
                

Updated User Profile:
{'films_liked': [], 'films_disliked': [], 'genres_positive': [], 'genres_negative': [], 'actors_positive': [], 'actors_negative': [], 'directors_positive': [], 'directors_negative': [], 'languages_positive': [], 'languages_negative': []}


In [15]:
# Sample user input
# user_input = "I love horror films and Sandra Bullock, but I can't stand Brad Pitt. I prefer movies in English. I don't like Chinese films. I don't like Brad Pitt.I don't like French films."
sample_user_input_feedback = "I hate Brad Pitt"


# Parse user input
parsed_data = parse_user_sentiment_input(sample_user_input_feedback)
print(parsed_data)

# initialize_user_profile()
# Update the user profile
user_profile = update_user_profile(user_profile, parsed_data)

print("Updated User Profile:")
print(user_profile)


{'films': [], 'genres': [], 'actors': ['Brad Pitt'], 'directors': [], 'languages': []}
Brad Pitt
Updated User Profile:
{'films_liked': [], 'films_disliked': [], 'genres_positive': [], 'genres_negative': [], 'actors_positive': [], 'actors_negative': [], 'directors_positive': [], 'directors_negative': [], 'languages_positive': [], 'languages_negative': []}


In [6]:
# Sample user input
# user_input = "I love horror films and Sandra Bullock, but I can't stand Brad Pitt. I prefer movies in English. I don't like Chinese films. I don't like Brad Pitt.I don't like French films."
sample_user_input_feedback = "I don't like Sandra Bullock"


# Parse user input
parsed_data = parse_user_sentiment_input(sample_user_input_feedback)

# Update the user profile
user_profile = update_user_profile(user_profile, parsed_data)

print("Updated User Profile:")
print(user_profile)


Updated User Profile:
{'films_liked': [], 'films_disliked': [], 'genres_positive': [], 'genres_negative': [], 'actors_positive': [], 'actors_negative': [], 'directors_positive': [], 'directors_negative': [], 'languages_positive': [], 'languages_negative': []}


In [7]:
# Sample questions to ask the user
questions = [
    "Have you seen any good movies recently?",
    "What are your favorite genres?",
    "Who are your favorite actors?",
    "Who are your favorite directors?",
    "Do you have any favorite languages for films?",
    "Are there any actors you dislike?",
    "Are there any directors you dislike?",
    "Are there any genres you dislike?",
    "Are there any languages you dislike?"
]

# Ask the user questions
for question in questions:
    user_question_input = input(question + " ")
    parsed_data = parse_user_sentiment_input(user_question_input)
    user_profile = update_user_profile(user_profile, parsed_data)
    
print("Final User Profile:")
print(user_profile)





KeyboardInterrupt: Interrupted by user